# 📈 Commodity Price Prediction - Daily Runner

This notebook fetches the latest commodity prices, runs predictions, and saves results.

**Options for automation:**
1. **Manual**: Run this notebook daily
2. **Colab Pro**: Use the scheduler feature
3. **GitHub Actions**: Recommended free option (see repo)

---

In [ ]:
# Install dependencies
!pip install -q yfinance torch pandas scikit-learn

In [ ]:
# Configuration
COMMODITIES = {
    'CL=F': 'Crude Oil WTI',
    'GC=F': 'Gold',
    'SI=F': 'Silver',
    'NG=F': 'Natural Gas',
    'HG=F': 'Copper',
    'ZC=F': 'Corn',
    'ZW=F': 'Wheat',
}

SEQUENCE_LENGTH = 60
PREDICTION_HORIZONS = [1, 5, 10]

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print(f"Running predictions at {datetime.now()}")

In [ ]:
# Model Definition
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [ ]:
def create_features(series):
    """Create technical features."""
    df = pd.DataFrame({'price': series})
    
    # Lags
    for lag in [1, 2, 3, 5, 7, 14, 21]:
        df[f'lag_{lag}'] = df['price'].shift(lag)
    
    # Returns
    for period in [1, 5, 10, 21]:
        df[f'return_{period}d'] = df['price'].pct_change(period)
    
    # Moving averages
    for window in [5, 10, 20, 50]:
        df[f'sma_{window}'] = df['price'].rolling(window).mean()
        df[f'price_to_sma_{window}'] = df['price'] / df[f'sma_{window}']
    
    # Volatility
    returns = df['price'].pct_change()
    for window in [5, 10, 21]:
        df[f'vol_{window}d'] = returns.rolling(window).std()
    
    # RSI
    delta = df['price'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))
    
    return df


def train_and_predict(series, seq_length=60):
    """Train model and make prediction."""
    # Create features
    df = create_features(series).dropna()
    
    if len(df) < seq_length + 50:
        return None
    
    # Scale
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)
    
    # Create sequences
    X, y = [], []
    for i in range(seq_length, len(scaled) - 1):
        X.append(scaled[i-seq_length:i])
        y.append(scaled[i+1, 0])
    
    X = np.array(X)
    y = np.array(y).reshape(-1, 1)
    
    # Train
    model = BiLSTMModel(X.shape[2])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    X_train = torch.FloatTensor(X[:-30])
    y_train = torch.FloatTensor(y[:-30])
    
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()
    
    # Predict
    model.eval()
    with torch.no_grad():
        last_seq = torch.FloatTensor(scaled[-seq_length:].reshape(1, seq_length, -1))
        pred_scaled = model(last_seq).numpy()[0, 0]
    
    # Inverse transform
    dummy = np.zeros((1, scaler.n_features_in_))
    dummy[0, 0] = pred_scaled
    pred_price = scaler.inverse_transform(dummy)[0, 0]
    
    return pred_price

In [ ]:
# Fetch data and make predictions
results = {
    'generated_at': datetime.now().isoformat(),
    'predictions': {},
    'historical': {}
}

for ticker, name in COMMODITIES.items():
    print(f"\nProcessing {name}...")
    
    try:
        data = yf.download(ticker, period='1y', progress=False)['Close']
        
        if len(data) < 100:
            print(f"  Insufficient data")
            continue
        
        current_price = float(data.iloc[-1])
        pred_price = train_and_predict(data)
        
        if pred_price is None:
            continue
        
        change_pct = (pred_price - current_price) / current_price * 100
        
        results['predictions'][name] = {
            'ticker': ticker,
            'current_price': round(current_price, 2),
            'last_updated': data.index[-1].strftime('%Y-%m-%d'),
            'forecasts': {
                '1d': {'price': round(pred_price, 2), 'change_pct': round(change_pct, 2)},
                '5d': {'price': round(current_price * (1 + change_pct/100 * np.sqrt(5)), 2), 
                       'change_pct': round(change_pct * np.sqrt(5), 2)},
                '10d': {'price': round(current_price * (1 + change_pct/100 * np.sqrt(10)), 2),
                        'change_pct': round(change_pct * np.sqrt(10), 2)}
            }
        }
        
        # Historical data for charts
        recent = data.tail(90)
        results['historical'][name] = {
            'dates': [d.strftime('%Y-%m-%d') for d in recent.index],
            'prices': [round(float(p), 2) for p in recent.values]
        }
        
        arrow = '↑' if change_pct > 0 else '↓'
        print(f"  Current: ${current_price:.2f} → Predicted: ${pred_price:.2f} ({arrow}{abs(change_pct):.1f}%)")
        
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# Save results
with open('latest_predictions.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "="*50)
print("Predictions saved to latest_predictions.json")
print("="*50)

# Display results
for name, data in results['predictions'].items():
    print(f"\n{name}:")
    print(f"  Current: ${data['current_price']}")
    for h, pred in data['forecasts'].items():
        print(f"  {h}: ${pred['price']} ({'+' if pred['change_pct'] > 0 else ''}{pred['change_pct']}%)")

In [ ]:
# Download the predictions file
from google.colab import files
files.download('latest_predictions.json')

## 📤 Upload to GitHub (Optional)

Uncomment and run the cell below to automatically push predictions to your GitHub repo.

In [ ]:
# # Uncomment to upload to GitHub
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# !cp latest_predictions.json YOUR_REPO/predictions/
# %cd YOUR_REPO
# !git config user.email "your-email@example.com"
# !git config user.name "Your Name"
# !git add predictions/latest_predictions.json
# !git commit -m "Update predictions $(date +'%Y-%m-%d')"
# !git push https://YOUR_TOKEN@github.com/YOUR_USERNAME/YOUR_REPO.git